# Python implementation of the Fast Adaptive Spectrum Sensing Algorithm
This notebook contains the python implementation of the fast adaptive spectrum sensing algorithm as detailed [here](https://ieeexplore.ieee.org/document/10058972)

## Fast Adaptive Spectrum Sensing State Machine
At the core of the Fast Adaptive Spectrum Sensing Algorithm (FSS) is the state machine defined below. This machine uses two buckets, filling the smaller bucket for each gap between high powered subbands on the spectrum. This essentially leaves the largest bucket after the processing is finished, where the buckets can be reset. 

In [ ]:
# define the state machine for the algorithm
class Transition:
    def __init__(self, from_state, to_state, condition, additional_actions=None):
        self.from_state = from_state
        self.to_state = to_state
        self.condition = condition
        self.additional_actions = additional_actions

In [ ]:
class FSSStateMachine:
    # define the states of the state machine
    SEARCH = 0
    FILL_BUCKET_1 = 1
    FILL_BUCKET_2 = 2

    def __init__(self):
        self.states = ["SEARCH", "FILL BUCKET 1", "FILL BUCKET 2"]
        self.current_state = 0

        # algorithm parameters (these are used and changed by the state machine)
        self.packet_count = 0
        self.b1_start = 0
        self.b1_size = 0
        self.b2_start = 0
        self.b2_size = 0
        self.threshold = 3.3063128e-09 # dB threshold for the FSS detection algorithm

        self.state_transitions = [[
            # ========== SEARCH STATE TRANSITION ==========
            Transition(
                0, 
                0, 
                (lambda x: x > self.threshold), 
                (self.s_to_s)
            ), 
            Transition(
                0, 
                1, 
                (lambda x: x <= self.threshold and self.b1_size <= self.b2_size), 
                (self.s_to_fb1)
            ),
            Transition(
                0, 
                2, 
                (lambda x: x <= self.threshold and self.b1_size > self.b2_size), 
                (self.s_to_fb2)
            )
        ], [
            # ========== FILL BUCKET 1 STATE TRANSITION ==========
            Transition(
                1, 
                1, 
                (lambda x: x <= self.threshold), 
                (self.fb1_to_fb1)
            ),
            Transition(
                1, 
                0, 
                (lambda x: x > self.threshold), 
                (self.fb1_to_s)
            )
        ], [
            # ========== FILL BUCKET 2 STATE TRANSITION ==========
            Transition(
                2, 
                2, 
                (lambda x: x <= self.threshold), 
                (self.fb2_to_fb2)
            ),
            Transition(
                2, 
                0, 
                (lambda x: x > self.threshold), 
                (self.fb2_to_s)
            )
        ]]

    # ========== SEARCH STATE TRANSITION ACTIONS ==========

    def s_to_s(self):
        self.packet_count += 1

    def s_to_fb1(self):
        self.b1_start = self.packet_count
        self.b1_size = 1
        self.packet_count += 1

    def s_to_fb2(self):
        self.b2_start = self.packet_count
        self.b2_size = 1
        self.packet_count += 1

    # ========== FILL BUCKET 1 STATE TRANSITION ACTIONS ==========

    def fb1_to_fb1(self):
        self.b1_size += 1
        self.packet_count += 1

    def fb1_to_s(self):
        self.packet_count += 1

    # ========== FILL BUCKET 2 STATE TRANSITION ACTIONS ==========
    def fb2_to_fb2(self):
        self.b2_size += 1
        self.packet_count += 1

    def fb2_to_s(self):
        self.packet_count += 1    

    # ========== STATE MACHINE METHODS ==========
    def cycle(self, input):
        current_state_transitions = self.state_transitions[self.current_state]

        for transition in current_state_transitions:

            if(transition.from_state != self.current_state):
                raise ValueError(f"Transition from state {transition.from_state} does not match current state {self.current_state}")

            if transition.condition(input):
                self.current_state = transition.to_state
                if transition.additional_actions:
                    transition.additional_actions()

                return

        raise ValueError(f"No valid transition found for input {input} in state {self.current_state}")

    # used in place of a reset flag
    def RESET(self):
        if self.b1_size >= self.b2_size:
            temp = (self.b1_start, self.b1_size)
        else:
            temp = (self.b2_start, self.b2_size)
        
        self.current_state = 0
        self.packet_count = 0
        self.b1_start = 0
        self.b1_size = 0
        self.b2_start = 0
        self.b2_size = 0

        return temp

## Using the FSS Algorithm
The following is the usage of the FSS algoirthm on data captured with the USRP X310. The algoirthm is general and can be applied anywhere on the frequency band. The catch of this algorithm is setting a threshold, which in the future will be set using the HO-CAE algorithm defined later in the paper.

### Preprocessing. 
Load, check, and apply a Short Time Fourier Transform to the IQ data.

In [ ]:
from utils.data import load_iq_data

FILE_NAME = "usrp_2.45g_test_capture.bin"

# Load the IQ data from the binary file
data = load_iq_data(FILE_NAME)
print(f"Loaded {len(data)} samples from {FILE_NAME}")

In [ ]:
# Get all of the constants of the data. 
# In the future we should store this as metadata alongside the .bin file, but for now we will hardcode it here.
SAMP_RATE = 100e6
CENTER_FREQ = 2.45e9
DURATION = len(data) / SAMP_RATE


In [ ]:
from utils.plot import plot_spectrogram

# plot a section of the data to understand what it looks like before the algorithm
data_to_plot = data[:int(SAMP_RATE * 0.01)] # use the first 10ms of the IQ data for plotting
plot_spectrogram(data_to_plot, SAMP_RATE, CENTER_FREQ)

In [ ]:
import numpy as np
import scipy.signal as signal
import matplotlib.pyplot as plt

# Ensure IQ data is a one-dimensional complex array
data = np.asarray(data[:int(SAMP_RATE*0.01)]).squeeze()

if data.ndim != 1:
    raise ValueError(f"Expected 1D IQ data, received shape {data.shape}")

# STFT parameters
NFFT = 1024
OVERLAP = 512

# Compute the short-time Fourier transform
f, t, z = signal.stft(
    data,
    fs=SAMP_RATE,
    window="hann",
    nperseg=NFFT,
    noverlap=OVERLAP,
    nfft=NFFT,
    return_onesided=False,
    boundary=None,
    padded=False,
)

# Put negative frequencies on the left and positive frequencies on the right
f_shifted = np.fft.fftshift(f)
z_shifted = np.fft.fftshift(z, axes=0)

# conver to magnitude squared for runing the algoirithm
mag_sq = np.abs(z_shifted) ** 2
mag_sq = mag_sq.T

# still grab the magnitude db for plotting spectrograms
magnitude_db = 20.0 * np.log10(np.maximum(np.abs(z_shifted), 1e-12))
magnitude_db = magnitude_db.T

print(f"Spectrogram shape: {mag_sq.shape}, Frequency bins: {len(f_shifted)}, Time bins: {len(t)}")

### Apply the algorithm and plot the results

In [ ]:
# run through the state machine a little bit to see it working i guess

fss_sm = FSSStateMachine()
bin_idx = 0
bin_data = []
for bin in mag_sq:
    for magnitude in bin:
        fss_sm.cycle(magnitude)
    bin_idx += 1

    bucket_data = fss_sm.RESET()
    print(f"Bin {bin_idx} - Bucket data: Start index = {bucket_data[0]}, Size = {bucket_data[1]}")
    bin_data.append(bucket_data)

In [ ]:
# plot the spectrogram with the detected buckets overlaid
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle

magnitude_db = np.asarray(magnitude_db)
f_shifted = np.asarray(f_shifted)
t = np.asarray(t)

print("magnitude_db:", magnitude_db.shape)
print("frequency bins:", len(f_shifted))
print("time bins:", len(t))

# Force C into shape:
# rows = time
# columns = frequency
if magnitude_db.shape == (len(f_shifted), len(t)):
    plot_data = magnitude_db.T
elif magnitude_db.shape == (len(t), len(f_shifted)):
    plot_data = magnitude_db
else:
    raise ValueError(
        f"Unexpected spectrogram shape {magnitude_db.shape}. "
        f"Expected {(len(f_shifted), len(t))} or "
        f"{(len(t), len(f_shifted))}."
    )

fig, ax = plt.subplots(figsize=(12, 6))

spectrogram = ax.pcolormesh(
    f_shifted,
    t,
    plot_data,
    shading="auto",
    cmap="gray",
)

df = np.median(np.diff(f_shifted))
dt = np.median(np.diff(t))

for time_idx, (start_bin, bucket_size) in enumerate(bin_data):
    if time_idx >= len(t):
        break

    start_bin = int(start_bin)
    bucket_size = int(bucket_size)

    if bucket_size <= 0:
        continue

    end_bin = start_bin + bucket_size - 1

    start_bin = np.clip(start_bin, 0, len(f_shifted) - 1)
    end_bin = np.clip(end_bin, 0, len(f_shifted) - 1)

    start_frequency = f_shifted[start_bin]
    end_frequency = f_shifted[end_bin] + df

    rectangle = Rectangle(
        (
            start_frequency,
            t[time_idx] - dt / 2,
        ),
        width=end_frequency - start_frequency,
        height=dt,
        facecolor="red",
        edgecolor="none",
        alpha=0.15,
    )

    ax.add_patch(rectangle)

ax.set_title("Spectrogram with Detected Buckets")
ax.set_xlabel("Frequency Offset [Hz]")
ax.set_ylabel("Time [s]")

fig.colorbar(
    spectrogram,
    ax=ax,
    label="Intensity [dB]",
)

plt.tight_layout()
plt.show()